# Component 1 — Adaptive Web Attack Detection & Correlation Model
## Notebook 03: Feature Preprocessing, Multi-Label Detection & Evaluation

**Student:** Weerasinghe W.G.O.N (IT23378148)  
**Research Group:** J26-CS-303  
**Supervisor:** Mr. Amila Senarathna  
**Component:** Component 1 (Web Attack Detection & Correlation)  

---
### Objective of this Notebook:
1. Load the clean training and testing partitions created in Phase 2.
2. Configure a scikit-learn `ColumnTransformer` (TF-IDF for text features, One-Hot Encoding for categorical features).
3. Fit the preprocessor **only on training data** to strictly prevent data leakage.
4. Train a **One-vs-Rest Logistic Regression** multi-label classifier.
5. Generate both binary predictions and continuous confidence probabilities for all 13 attack categories.
6. Evaluate the model using Micro/Macro Precision, Recall, F1-score, and Hamming Loss.
7. Perform error analysis across majority and minority attack categories.
8. Export `correlation_input.csv` as the formal handover artifact for Stage 2 (PP2).

In [1]:
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report
)

# Add src to python path for modular imports
current_dir = Path.cwd()
if str(current_dir.parent) not in sys.path:
    sys.path.append(str(current_dir.parent))

from src.config import (
    X_TRAIN_PATH,
    X_TEST_PATH,
    Y_TRAIN_PATH,
    Y_TEST_PATH,
    CONTEXT_TEST_PATH,
    CATEGORICAL_FEATURES,
    TEXT_FEATURES,
    MODEL_LABEL_COLUMNS,
    RANDOM_STATE,
    MODEL_PATH,
    PREPROCESSOR_PATH,
    DETECTION_OUTPUT_CSV,
    CORRELATION_INPUT_CSV,
    RESULTS_DIR
)

print("Environment initialized successfully.")

Environment initialized successfully.


## 1. Load Clean Partitions

We load the clean partitions generated from Phase 2 using `joblib`.

In [2]:
X_train = joblib.load(X_TRAIN_PATH)
X_test = joblib.load(X_TEST_PATH)
y_train = joblib.load(Y_TRAIN_PATH)
y_test = joblib.load(Y_TEST_PATH)
context_test = joblib.load(CONTEXT_TEST_PATH)

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test  shape: {y_test.shape}")
print(f"context_test shape: {context_test.shape}")

X_train shape: (724054, 14)
X_test  shape: (181014, 14)
y_train shape: (724054, 13)
y_test  shape: (181014, 13)
context_test shape: (181014, 5)


## 2. Feature Transformation Pipeline

We configure feature transformers for each data type:
- **Text Features (`request_http_request`, `request_body`):** Transformed using TF-IDF (`max_features=10,000`) to capture tokenized attack payloads.
- **Categorical Features (12 HTTP headers & methods):** Transformed using One-Hot Encoding (`handle_unknown='ignore'`).

In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "request_tfidf",
            TfidfVectorizer(max_features=10000),
            "request_http_request"
        ),
        (
            "body_tfidf",
            TfidfVectorizer(max_features=10000),
            "request_body"
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES
        )
    ]
)

print("ColumnTransformer preprocessor configured successfully.")

ColumnTransformer preprocessor configured successfully.


## 3. Fit Preprocessor on Training Data & Transform

To prevent data leakage, `preprocessor.fit_transform()` is executed strictly on `X_train`. `X_test` is only passed through `.transform()`.

In [4]:
t0 = time.time()
print("Fitting preprocessor on X_train...")
X_train_transformed = preprocessor.fit_transform(X_train)
X_train_transformed.sort_indices()

print("Transforming X_test...")
X_test_transformed = preprocessor.transform(X_test)
X_test_transformed.sort_indices()

print(f"Preprocessing completed in {time.time() - t0:.2f}s.")
print(f"X_train_transformed shape: {X_train_transformed.shape}")
print(f"X_test_transformed shape : {X_test_transformed.shape}")

Fitting preprocessor on X_train...


Transforming X_test...


Preprocessing completed in 27.97s.
X_train_transformed shape: (724054, 34822)
X_test_transformed shape : (181014, 34822)


## 4. One-vs-Rest Logistic Regression Classifier

We train an independent binary Logistic Regression classifier for each of the 13 attack labels using the `liblinear` solver with `n_jobs=1` for deterministic, memory-safe execution.

In [5]:
base_classifier = LogisticRegression(
    solver="liblinear",
    max_iter=1000,
    random_state=RANDOM_STATE
)
model = OneVsRestClassifier(base_classifier, n_jobs=1)

print("Training One-vs-Rest Logistic Regression model...")
t_train = time.time()
model.fit(X_train_transformed, y_train)
print(f"Model trained successfully in {time.time() - t_train:.2f}s.")
print(f"Number of trained binary classifiers: {len(model.estimators_)}")

Training One-vs-Rest Logistic Regression model...


Model trained successfully in 196.03s.
Number of trained binary classifiers: 13


## 5. Generate Predictions and Probability Estimates

We predict discrete binary labels as well as continuous probability estimates for all 13 attack categories on the held-out test data.

In [6]:
y_pred = model.predict(X_test_transformed)
y_prob = model.predict_proba(X_test_transformed)

print(f"Predictions generated: {y_pred.shape}")
print(f"Probabilities generated: {y_prob.shape}")

Predictions generated: (181014, 13)
Probabilities generated: (181014, 13)


## 6. Overall Multi-Label Performance Evaluation

We calculate Micro-averaged metrics (global performance), Macro-averaged metrics (unweighted class average), and Hamming Loss (fraction of incorrectly predicted labels).

In [7]:
micro_p = precision_score(y_test, y_pred, average="micro", zero_division=0)
micro_r = recall_score(y_test, y_pred, average="micro", zero_division=0)
micro_f1 = f1_score(y_test, y_pred, average="micro", zero_division=0)

macro_p = precision_score(y_test, y_pred, average="macro", zero_division=0)
macro_r = recall_score(y_test, y_pred, average="macro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

h_loss = hamming_loss(y_test, y_pred)

print("Overall Multi-Label Model Performance")
print("=" * 40)
print(f"Micro Precision : {micro_p:.4f}")
print(f"Micro Recall    : {micro_r:.4f}")
print(f"Micro F1-Score  : {micro_f1:.4f}")
print("-" * 40)
print(f"Macro Precision : {macro_p:.4f}")
print(f"Macro Recall    : {macro_r:.4f}")
print(f"Macro F1-Score  : {macro_f1:.4f}")
print("-" * 40)
print(f"Hamming Loss    : {h_loss:.4f}")

Overall Multi-Label Model Performance
Micro Precision : 0.9877
Micro Recall    : 0.9831
Micro F1-Score  : 0.9854
----------------------------------------
Macro Precision : 0.9800
Macro Recall    : 0.8893
Macro F1-Score  : 0.9227
----------------------------------------
Hamming Loss    : 0.0023


## 7. Per-Label Performance Breakdown

We inspect Precision, Recall, and F1-score for each of the 13 attack categories individually.

In [8]:
report = classification_report(
    y_test,
    y_pred,
    target_names=MODEL_LABEL_COLUMNS,
    output_dict=True,
    zero_division=0
)
per_label_df = pd.DataFrame(report).transpose().loc[MODEL_LABEL_COLUMNS, ["precision", "recall", "f1-score", "support"]]
display(per_label_df)

,precision,recall,f1-score,support
000 - Normal,0.987789,0.993809,0.990790,104352.0
272 - Protocol Manipulation,0.853367,0.554878,0.672489,1804.0
242 - Code Injection,0.995551,0.976012,0.985685,3210.0
88 - OS Command Injection,0.975538,0.641570,0.774068,1554.0
126 - Path Traversal,0.985229,0.938335,0.961211,4265.0
66 - SQL Injection,0.990062,0.990062,0.990062,50109.0
16 - Dictionary-based Password Attack,1.000000,1.000000,1.000000,382.0
310 - Scanning for Vulnerable Software,0.990706,0.998127,0.994403,534.0
153 - Input Data Manipulation,0.993421,1.000000,0.996700,453.0
274 - HTTP Verb Tampering,1.000000,1.000000,1.000000,1132.0


## 8. Minority Class Under-Detection Analysis

Due to severe class imbalance, some minority classes have lower recall. We quantify the difference between actual positives and predicted positives.

In [9]:
actual_positives = y_test.sum(axis=0)
predicted_positives = y_pred.sum(axis=0)

error_analysis = pd.DataFrame({
    "Actual Positives": actual_positives,
    "Predicted Positives": predicted_positives,
    "Difference": predicted_positives - actual_positives
}).sort_values("Actual Positives", ascending=False)

display(error_analysis)

,Actual Positives,Predicted Positives,Difference
000 - Normal,104352,104988,636
66 - SQL Injection,50109,50109,0
194 - Fake the Source of Data,11173,11192,19
126 - Path Traversal,4265,4062,-203
34 - HTTP Response Splitting,3915,3941,26
242 - Code Injection,3210,3147,-63
272 - Protocol Manipulation,1804,1173,-631
88 - OS Command Injection,1554,1022,-532
274 - HTTP Verb Tampering,1132,1132,0
310 - Scanning for Vulnerable Software,534,538,4


## 9. PP1-to-PP2 Handover Artifact: `correlation_input.csv`

We assemble the final deliverable of the Detection Stage: the 5 preserved contextual fields combined with the 13 binary predictions and 13 probability columns (31 columns total).

In [10]:
y_pred_df = pd.DataFrame(y_pred, index=X_test.index, columns=MODEL_LABEL_COLUMNS)
prob_cols = [f"{col} - Probability" for col in MODEL_LABEL_COLUMNS]
y_prob_df = pd.DataFrame(y_prob, index=X_test.index, columns=prob_cols)

detection_output = pd.concat([y_pred_df, y_prob_df], axis=1)
correlation_input = pd.concat([context_test, detection_output], axis=1)

correlation_input.to_csv(CORRELATION_INPUT_CSV, index=True)
print(f"Correlation input saved to: {CORRELATION_INPUT_CSV}")
print(f"Dimensions: {correlation_input.shape[0]:,} rows x {correlation_input.shape[1]} columns")
display(correlation_input.head())

Correlation input saved to: C:\Users\Meishaaa\OneDrive - Sri Lanka Institute of Information Technology\Desktop updates\Desktop\RA 1\results\correlation_input.csv
Dimensions: 181,014 rows x 31 columns


,timestamp,src_ip,src_port,dst_ip,dst_port,000 - Normal,272 - Protocol Manipulation,242 - Code Injection,88 - OS Command Injection,126 - Path Traversal,...,88 - OS Command Injection - Probability,126 - Path Traversal - Probability,66 - SQL Injection - Probability,16 - Dictionary-based Password Attack - Probability,310 - Scanning for Vulnerable Software - Probability,153 - Input Data Manipulation - Probability,274 - HTTP Verb Tampering - Probability,194 - Fake the Source of Data - Probability,34 - HTTP Response Splitting - Probability,33 - HTTP Request Smuggling - Probability
568169,21/Jul/2020:22:53:19 +0100,172.26.0.1,49783.0,172.26.0.4,80.0,1,0,0,0,0,...,0.000004,0.000008,0.000017,0.000001,0.000006,3.571663e-07,0.000002,6.601769e-06,0.000005,1.305479e-06
443732,18/Jul/2020:17:03:45 +0200,172.26.0.1,50088.0,172.26.0.4,80.0,0,0,0,0,0,...,0.000003,0.000008,0.999843,0.000004,0.000002,1.030848e-05,0.000006,4.450237e-07,0.000015,4.114344e-06
798813,29/Jul/2020:13:01:36 +0100,172.26.0.1,50422.0,172.26.0.4,80.0,1,0,0,0,0,...,0.000004,0.000008,0.000017,0.000001,0.000006,3.571746e-07,0.000002,6.602769e-06,0.000005,1.305560e-06
669662,29/Jul/2020:09:22:15 +0100,172.26.0.1,63696.0,172.26.0.4,80.0,1,0,0,0,0,...,0.000004,0.000008,0.000017,0.000001,0.000006,3.571668e-07,0.000002,6.602625e-06,0.000005,1.305500e-06
247253,18/Jul/2020:03:38:27 +0200,172.26.0.1,63017.0,172.26.0.4,80.0,0,0,0,0,0,...,0.000012,0.000039,0.937969,0.000001,0.000007,5.818160e-06,0.000032,9.470557e-04,0.001128,5.673563e-07
